In [1]:
import re
import pandas as pd

In [2]:

log_path = r"C:\Users\bernd\Documents\MP\OpenPCDet\output\OpenPCDet\tools\cfgs\models\P\D\P_D_K\default\eval\epoch_7728\val\default\log_eval_20260806-154046.txt"
with open(log_path) as f:
    text = f.read()

pattern = re.compile(
    r'(\w+)\s+(AP_R40@[^:\n]+|AP@[^:\n]+):\n'
    r'bbox AP:([^\n]+)\n'
    r'bev\s+AP:([^\n]+)\n'
    r'3d\s+AP:([^\n]+)',
    re.MULTILINE,
)

def parse_triplet(value_text):
    return [float(x.strip()) for x in value_text.split(',')]

results = {}
rows = []

for match in pattern.finditer(text):
    cls = match.group(1)
    setting = match.group(2)

    bbox_vals = parse_triplet(match.group(3))
    bev_vals = parse_triplet(match.group(4))
    d3_vals = parse_triplet(match.group(5))

    results.setdefault(cls, {})[setting] = {
        'bbox': bbox_vals,
        'bev': bev_vals,
        '3d': d3_vals,
    }

    rows.append({
        'class': cls,
        'setting': setting,
        'metric': 'bbox',
        'easy': bbox_vals[0],
        'moderate': bbox_vals[1],
        'hard': bbox_vals[2],
    })
    rows.append({
        'class': cls,
        'setting': setting,
        'metric': 'bev',
        'easy': bev_vals[0],
        'moderate': bev_vals[1],
        'hard': bev_vals[2],
    })
    rows.append({
        'class': cls,
        'setting': setting,
        'metric': '3d',
        'easy': d3_vals[0],
        'moderate': d3_vals[1],
        'hard': d3_vals[2],
    })

df = pd.DataFrame(rows)
print(df)
print({cls: list(settings.keys()) for cls, settings in results.items()})

         class                  setting metric     easy  moderate     hard
0          Car      AP@0.70, 0.70, 0.70   bbox  73.0237   74.1605  73.6345
1          Car      AP@0.70, 0.70, 0.70    bev  72.5963   72.5642  70.5633
2          Car      AP@0.70, 0.70, 0.70     3d  70.0393   65.1719  63.1360
3          Car  AP_R40@0.70, 0.70, 0.70   bbox  74.5727   74.8080  74.3508
4          Car  AP_R40@0.70, 0.70, 0.70    bev  73.0180   72.2189  71.3203
5          Car  AP_R40@0.70, 0.70, 0.70     3d  70.0331   64.9059  62.4729
6          Car      AP@0.70, 0.50, 0.50   bbox  73.0237   74.1605  73.6345
7          Car      AP@0.70, 0.50, 0.50    bev  73.6399   74.9681  74.6706
8          Car      AP@0.70, 0.50, 0.50     3d  73.6399   74.8749  74.5303
9          Car  AP_R40@0.70, 0.50, 0.50   bbox  74.5727   74.8080  74.3508
10         Car  AP_R40@0.70, 0.50, 0.50    bev  75.3157   77.1240  76.8682
11         Car  AP_R40@0.70, 0.50, 0.50     3d  75.3146   76.8834  76.4374
12  Pedestrian      AP@0.

In [3]:
model = "SECOND Default"
finetuned = "No"
dataset = "nuScenes"
target_setting = "AP_R40@0.70, 0.70, 0.70"


def pick_setting(class_results, preferred_setting):
    if preferred_setting in class_results:
        return preferred_setting

    ap_r40_settings = [setting for setting in class_results if setting.startswith('AP_R40@')]
    if not ap_r40_settings:
        raise KeyError('No AP_R40 settings found for this class.')

    return sorted(ap_r40_settings)[0]


print(results)
print(f"Using setting: {target_setting}")



{'Car': {'AP@0.70, 0.70, 0.70': {'bbox': [73.0237, 74.1605, 73.6345], 'bev': [72.5963, 72.5642, 70.5633], '3d': [70.0393, 65.1719, 63.136]}, 'AP_R40@0.70, 0.70, 0.70': {'bbox': [74.5727, 74.808, 74.3508], 'bev': [73.018, 72.2189, 71.3203], '3d': [70.0331, 64.9059, 62.4729]}, 'AP@0.70, 0.50, 0.50': {'bbox': [73.0237, 74.1605, 73.6345], 'bev': [73.6399, 74.9681, 74.6706], '3d': [73.6399, 74.8749, 74.5303]}, 'AP_R40@0.70, 0.50, 0.50': {'bbox': [74.5727, 74.808, 74.3508], 'bev': [75.3157, 77.124, 76.8682], '3d': [75.3146, 76.8834, 76.4374]}}, 'Pedestrian': {'AP@0.50, 0.50, 0.50': {'bbox': [49.4412, 48.1426, 46.0142], 'bev': [47.6378, 44.5363, 41.8414], '3d': [44.501, 41.6599, 38.4552]}, 'AP_R40@0.50, 0.50, 0.50': {'bbox': [47.6709, 46.1766, 43.7748], 'bev': [45.2082, 42.2924, 39.3234], '3d': [41.99, 38.8809, 35.6762]}, 'AP@0.50, 0.25, 0.25': {'bbox': [49.4412, 48.1426, 46.0142], 'bev': [54.1623, 53.3491, 50.5705], '3d': [54.1388, 53.1366, 50.4579]}, 'AP_R40@0.50, 0.25, 0.25': {'bbox': [47.

In [4]:
cells = []
chosen_setting = target_setting

for cls in ["Car", "Pedestrian", "Cyclist"]:
    class_results = results[cls]
    setting = pick_setting(class_results, chosen_setting)
    bev = class_results[setting]["bev"]
    d3 = class_results[setting]["3d"]

    for i in range(3):
        cells.append(f"{bev[i]:.4f}/{d3[i]:.4f}")

latex_row = (
    f"{model} & {finetuned} & {dataset} & "
    + " & ".join(cells)
    + r" \\")

print(latex_row)

SECOND Default & No & nuScenes & 73.0180/70.0331 & 72.2189/64.9059 & 71.3203/62.4729 & 52.7712/52.7473 & 52.0749/51.8193 & 49.2129/48.9384 & 71.1025/71.1025 & 56.2304/56.2304 & 53.0714/53.0714 \\


In [ ]:
benchmarks = {
    "Car": "AP_R40@0.70, 0.70, 0.70",
    "Pedestrian": "AP_R40@0.50, 0.50, 0.50",
    "Cyclist": "AP_R40@0.50, 0.50, 0.50",
}
relaxed = {
    "Car": "AP_R40@0.70, 0.50, 0.50",
    "Pedestrian": "AP_R40@0.50, 0.25, 0.25",
    "Cyclist": "AP_R40@0.50, 0.25, 0.25",
}

benchmark_cells = []
relaxed_cells = []

for cls in ["Car", "Pedestrian", "Cyclist"]:
    class_results = results[cls]
    # pick the setting string from the benchmarks dict for this class
    setting = pick_setting(class_results, benchmarks[cls])
    bev = class_results[setting]["bev"]
    d3 = class_results[setting]["3d"]
    for i in range(3):
        benchmark_cells.append(f"{bev[i]:.4f}/{d3[i]:.4f}")

for cls in ["Car", "Pedestrian", "Cyclist"]:
    class_results = results[cls]
    setting = pick_setting(class_results, relaxed[cls])
    bev = class_results[setting]["bev"]
    d3 = class_results[setting]["3d"]

    for i in range(3):
        relaxed_cells.append(f"{bev[i]:.4f}/{d3[i]:.4f}")

benchmark_row = " & ".join(benchmark_cells) + r" \\"
relaxed_row = " & ".join(relaxed_cells) + r" \\"

print(benchmark_row)
print(relaxed_row)



73.0180/70.0331 & 72.2189/64.9059 & 71.3203/62.4729 & 45.2082/41.9900 & 42.2924/38.8809 & 39.3234/35.6762 & 68.7673/65.7615 & 52.9946/50.1757 & 49.8275/46.9627 \\
75.3157/75.3146 & 77.1240/76.8834 & 76.8682/76.4374 & 75.3157/75.3146 & 77.1240/76.8834 & 76.8682/76.4374 & 75.3157/75.3146 & 77.1240/76.8834 & 76.8682/76.4374 & 52.7712/52.7473 & 52.0749/51.8193 & 49.2129/48.9384 & 52.7712/52.7473 & 52.0749/51.8193 & 49.2129/48.9384 & 52.7712/52.7473 & 52.0749/51.8193 & 49.2129/48.9384 & 71.1025/71.1025 & 56.2304/56.2304 & 53.0714/53.0714 & 71.1025/71.1025 & 56.2304/56.2304 & 53.0714/53.0714 & 71.1025/71.1025 & 56.2304/56.2304 & 53.0714/53.0714 \\
